# 02 Marts

สร้างตารางสรุปจาก fact table (`data/parquet/od/`) เก็บไว้ที่ `data/marts/` ให้ notebook ต่อๆ ไปใช้

ตารางที่ได้
- `stations` ข้อมูลสถานีและวันเปิดจริง
- `calendar` ปฏิทินรายวัน วันหยุด ช่วง COVID และสถานะของวัน (day_status)
- `station_hour` ผู้โดยสารเข้า/ออก รายสถานี รายชั่วโมง (เติมศูนย์ครบ)
- `station_day` ผู้โดยสารเข้า/ออก รายสถานี รายวัน
- `system_hour` ทริปทั้งระบบ รายชั่วโมง
- `od_month` ทริปรายคู่สถานี รายเดือน แยกวันทำงานกับวันหยุด
- `month_days` จำนวนวันในแต่ละเดือน ใช้หาค่าเฉลี่ยต่อวันของ od_month

กฎการเติมข้อมูล
- feed ไม่เก็บแถวที่มี 0 ทริป ชั่วโมงที่ไม่มีแถวจึงเติม 0
- วันที่ feed ไม่มีข้อมูลเลย (no_data) ใส่ค่าว่าง (NULL) ไม่ใส่ 0 เพราะไม่รู้ค่าจริง
- ก่อนวันเปิดสถานีไม่สร้างแถว เพราะสถานียังไม่มี
- ทริปที่เข้าและออกสถานีเดียวกันนับรวมใน entries/exits เพราะผ่านประตูจริง
- exits ใช้ชั่วโมงเดียวกับในข้อมูล (feed มีชั่วโมงเดียวต่อทริป ไม่รู้เวลาออกจริง)

In [1]:
import sys
from datetime import timedelta
from pathlib import Path

if Path.cwd().name == "notebooks":
    ROOT = Path.cwd().parent
else:
    ROOT = Path.cwd()
sys.path.append(str(ROOT / "src"))

import holidays
import pandas as pd
import plotly.graph_objects as go

from bart import viz
from bart.config import MARTS, OD, duck, fetch_row, fetch_value
from bart.perf import stage
from bart.stations import STATIONS

viz.notebook_mode()

In [2]:
con = duck()

con.execute(f"""
    CREATE OR REPLACE VIEW od AS
    SELECT date, hour, origin, dest, trips
    FROM read_parquet('{OD}/*/*.parquet')
""")

fact_rows, fact_trips = fetch_row(con, "SELECT count(*), sum(trips) FROM od")
first_date, last_date = fetch_row(con, "SELECT min(date), max(date) FROM od")

print(f"fact table: {fact_rows:,} rows, {fact_trips:,} trips, {first_date} to {last_date}")

fact table: 112,520,054 rows, 1,194,454,788 trips, 2011-01-01 to 2023-05-02


## stations

วันเปิดสถานีหาจากข้อมูล คือวันแรกที่มีผู้โดยสารเข้าสถานีอย่างน้อย 100 คน
เพราะสถานีใหม่มีทริปหลงอยู่ก่อนวันเปิดจริง (น่าจะเป็นการทดสอบระบบหรือพนักงาน)
สถานีที่เปิดก่อนปี 2011 จะได้วันเปิดเป็นวันแรกของข้อมูล

In [ ]:
station_rows = []
for code, (name, corridor, side) in STATIONS.items():
    station_rows.append({"station": code, "name": name, "corridor": corridor, "side": side})
station_info = pd.DataFrame(station_rows)

con.register("station_info", station_info)

con.execute("""
    CREATE OR REPLACE TEMP TABLE stations AS
    WITH daily_entries AS (
        SELECT origin AS station, date, sum(trips) AS entries
        FROM od
        GROUP BY origin, date
    ),
    first_dates AS (
        SELECT
            station,
            min(date)                               AS first_seen,
            min(date) FILTER (WHERE entries >= 100) AS opening_date
        FROM daily_entries
        GROUP BY station
    )
    SELECT
        i.station,
        i.name,
        i.corridor,
        i.side,
        f.first_seen,
        f.opening_date
    FROM station_info AS i
    JOIN first_dates AS f ON f.station = i.station
    ORDER BY i.station
""")

station_count = fetch_value(con, "SELECT count(*) FROM stations")
print("stations:", station_count)
assert station_count == 50

stations: 50


เทียบวันเปิดที่หาได้กับวันเปิดทางการของ BART

In [4]:
official_opening = {
    "WDUB": "2011-02-19",
    "OAKL": "2014-11-22",
    "WARM": "2017-03-25",
    "ANTC": "2018-05-26",
    "PCTR": "2018-05-26",
    "MLPT": "2020-06-13",
    "BERY": "2020-06-13",
}

new_stations = con.sql(f"""
    SELECT station, name, first_seen, opening_date
    FROM stations
    WHERE opening_date > DATE '{first_date}'
    ORDER BY opening_date
""").df()

official_dates = []
for station in new_stations["station"]:
    official_dates.append(pd.Timestamp(official_opening[station]))
new_stations["official_opening"] = official_dates
new_stations["match"] = new_stations["opening_date"] == new_stations["official_opening"]

new_stations

,station,name,first_seen,opening_date,official_opening,match
0,WDUB,West Dublin/Pleasanton,2011-02-02,2011-02-19,2011-02-19,True
1,OAKL,Oakland Airport,2014-09-29,2014-11-22,2014-11-22,True
2,WARM,Warm Springs/South Fremont,2016-06-23,2017-03-25,2017-03-25,True
3,ANTC,Antioch,2018-03-23,2018-05-26,2018-05-26,True
4,PCTR,Pittsburg Center,2018-04-17,2018-05-26,2018-05-26,True
5,BERY,Berryessa/North San Jose,2019-07-22,2020-06-13,2020-06-13,True
6,MLPT,Milpitas,2019-07-23,2020-06-13,2020-06-13,True


In [5]:
opened_after_start = len(new_stations)
all_match = bool(new_stations["match"].all())

print("stations opened after the data starts:", opened_after_start)
print("all opening dates match the official dates:", all_match)

assert opened_after_start == len(official_opening)
assert all_match

stations opened after the data starts: 7
all opening dates match the official dates: True


ทริปที่ใช้ต่อจากนี้ (valid_trips) ต้องเป็นทริปที่ทั้งสถานีต้นทางและปลายทางเปิดแล้ว
ดูว่าตัดทริปหลงออกไปเท่าไร

In [6]:
con.execute("""
    CREATE OR REPLACE VIEW valid_trips AS
    SELECT t.date, t.hour, t.origin, t.dest, t.trips
    FROM od AS t
    JOIN stations AS o ON o.station = t.origin
    JOIN stations AS d ON d.station = t.dest
    WHERE t.date >= o.opening_date
      AND t.date >= d.opening_date
""")

valid_rows, valid_trips = fetch_row(con, "SELECT count(*), sum(trips) FROM valid_trips")
stray_rows = fact_rows - valid_rows
stray_trips = fact_trips - valid_trips

print(f"valid trips : {valid_rows:,} rows, {valid_trips:,} trips")
print(f"stray trips : {stray_rows:,} rows, {stray_trips:,} trips ({stray_trips / fact_trips:.5%})")

valid trips : 112,519,592 rows, 1,194,453,535 trips
stray trips : 462 rows, 1,253 trips (0.00010%)


## วันหยุด

ใช้วันหยุดราชการ US จากไลบรารี `holidays` แต่ไม่ใช่ทุกวันที่คนหยุดจริง
ดูจากข้อมูลก่อน COVID ว่าวันหยุดแต่ละแบบมีผู้โดยสารเท่าไร เทียบกับค่ามัธยฐานของวันเดียวกันในสัปดาห์ช่วง ±4 สัปดาห์ (ไม่นับวันหยุด)
ใส่ Day after Thanksgiving เพิ่มเข้าไปด้วย เพราะหลายบริษัทหยุด

In [7]:
holiday_rows = []
for date, name in holidays.US(years=range(first_date.year, last_date.year + 1)).items():
    holiday_rows.append({"date": date, "name": name.replace(" (observed)", "")})

    if name == "Thanksgiving Day":
        day_after = date + timedelta(days=1)
        holiday_rows.append({"date": day_after, "name": "Day after Thanksgiving"})

all_holidays = pd.DataFrame(holiday_rows)
all_holidays["date"] = pd.to_datetime(all_holidays["date"])
con.register("all_holidays", all_holidays)

print(len(all_holidays), "holiday dates")

163 holiday dates


In [8]:
con.execute("""
    CREATE OR REPLACE TEMP TABLE daily AS
    SELECT date, sum(trips) AS day_trips
    FROM valid_trips
    GROUP BY date
""")

holiday_check = con.sql("""
    WITH normal_days AS (
        SELECT d.date, d.day_trips
        FROM daily AS d
        ANTI JOIN all_holidays AS h ON h.date = d.date
    ),
    ratios AS (
        SELECT
            h.name,
            d.day_trips / median(n.day_trips) AS ratio
        FROM all_holidays AS h
        JOIN daily AS d ON d.date = h.date
        JOIN normal_days AS n
            ON dayofweek(n.date) = dayofweek(d.date)
           AND n.date BETWEEN d.date - 28 AND d.date + 28
           AND n.date <> d.date
        WHERE d.date < DATE '2020-03-01'
          AND dayofweek(d.date) BETWEEN 1 AND 5
          AND d.date NOT BETWEEN DATE '2013-07-01' AND DATE '2013-07-05'
        GROUP BY h.name, d.date, d.day_trips
    )
    SELECT
        name,
        count(*)                AS weekdays,
        round(median(ratio), 2) AS median_ratio,
        round(min(ratio), 2)    AS min_ratio,
        round(max(ratio), 2)    AS max_ratio
    FROM ratios
    GROUP BY name
    ORDER BY median_ratio
""").df()

holiday_check

,name,weekdays,median_ratio,min_ratio,max_ratio
0,Christmas Day,9,0.17,0.12,0.43
1,Thanksgiving Day,9,0.19,0.15,0.24
2,Labor Day,9,0.29,0.20,0.51
3,New Year's Day,9,0.34,0.23,0.40
4,Memorial Day,9,0.36,0.26,0.41
5,Independence Day,8,0.39,0.29,0.56
6,Day after Thanksgiving,9,0.48,0.37,0.58
7,Washington's Birthday,10,0.50,0.40,0.56
8,Martin Luther King Jr. Day,10,0.54,0.41,0.59
9,Veterans Day,9,0.77,0.73,0.84


ratio คือผู้โดยสารวันนั้นเทียบกับวันธรรมดาปกติ ถ้าใกล้ 1 แปลว่าคนยังไปทำงานตามปกติ

- Columbus Day และ Veterans Day มีผู้โดยสารประมาณ 75-90% ของวันปกติ ส่วนใหญ่ยังทำงาน จึงไม่นับเป็นวันหยุด
- Juneteenth เพิ่งเป็นวันหยุดปี 2021 ไม่มีข้อมูลก่อน COVID ให้เทียบ ดูแยกใน cell ถัดไป
- วันอื่นผู้โดยสารลดลงชัดเจน รวมถึง Day after Thanksgiving นับเป็นวันหยุด

ช่วงคริสต์มาสถึงปีใหม่ (24-31 ธ.ค.) ผู้โดยสารก็ลดลงเหมือนกัน แต่ยังนับเป็นวันทำงาน แล้วค่อยระวังตอนวิเคราะห์ช่วงนั้น

In [9]:
con.sql("""
    SELECT
        d.date,
        dayname(d.date) AS day_name,
        d.day_trips,
        round(d.day_trips / median(n.day_trips), 2) AS ratio
    FROM all_holidays AS h
    JOIN daily AS d ON d.date = h.date
    JOIN daily AS n
        ON dayofweek(n.date) = dayofweek(d.date)
       AND n.date BETWEEN d.date - 28 AND d.date + 28
       AND n.date <> d.date
    WHERE h.name LIKE 'Juneteenth%'
    GROUP BY d.date, d.day_trips
    ORDER BY d.date
""").df()

,date,day_name,day_trips,ratio
0,2021-06-18,Friday,77495.0,0.97
1,2021-06-19,Saturday,49652.0,1.03
2,2022-06-19,Sunday,61281.0,0.98
3,2022-06-20,Monday,192757.0,1.56


Juneteenth ผู้โดยสารไม่ได้ต่ำกว่าวันปกติ (ratio ไม่ต่ำกว่า 0.97 ทุกปี) จึงไม่นับเป็นวันหยุด

In [10]:
not_holidays = ["Columbus Day", "Veterans Day", "Juneteenth National Independence Day"]

kept_rows = []
for _, holiday in all_holidays.iterrows():
    if holiday["name"] not in not_holidays:
        kept_rows.append(holiday)
holiday_list = pd.DataFrame(kept_rows).reset_index(drop=True)
con.register("holiday_list", holiday_list)

print(len(holiday_list), "holidays used")
print(sorted(holiday_list["name"].unique()))

128 holidays used
['Christmas Day', 'Day after Thanksgiving', 'Independence Day', 'Labor Day', 'Martin Luther King Jr. Day', 'Memorial Day', "New Year's Day", 'Thanksgiving Day', "Washington's Birthday"]


## calendar

day_status ของแต่ละวัน
- `no_data` feed ไม่มีข้อมูลทั้งวัน
- `strike` BART หยุดงานประท้วง 2013-07-01 ถึง 07-05 และ 2013-10-18 ถึง 10-21
- `partial` ข้อมูลน่าจะไม่ครบ ใช้ 2 กฎ
    - มีชั่วโมงในช่วง 7:00-20:00 ที่ไม่มีทริปเลยทั้งระบบ ซึ่งปกติไม่เกิดขึ้น
    - ไม่ใช่วันหยุด และทริปทั้งวันน้อยกว่า 50% ของทั้งสัปดาห์ก่อนและสัปดาห์หลัง (วันเดียวกันในสัปดาห์)
- `normal` วันปกติ

กฎข้อ 2 ใช้ทั้งสัปดาห์ก่อนและหลัง เพื่อไม่ให้ช่วงที่ผู้โดยสารลดลงจริงอย่างต่อเนื่อง (เช่น ต้นช่วง COVID) ถูกนับเป็นข้อมูลไม่ครบ

ช่วง COVID แบ่งเป็น pre (ก่อน 2020-03-09), break (2020-03-09 ถึง 2020-05-31) และ post (ตั้งแต่ 2020-06-01)

In [11]:
strike_days = [
    "2013-07-01", "2013-07-02", "2013-07-03", "2013-07-04", "2013-07-05",
    "2013-10-18", "2013-10-19", "2013-10-20", "2013-10-21",
]

strike_list = []
for day in strike_days:
    strike_list.append(f"DATE '{day}'")
strike_sql = ", ".join(strike_list)

In [12]:
calendar_query = f"""
    CREATE OR REPLACE TEMP TABLE calendar AS
    WITH dates AS (
        SELECT range::DATE AS date
        FROM range(DATE '{first_date}', DATE '{last_date}' + 1, INTERVAL 1 DAY)
    ),
    service_hours AS (
        SELECT date, hour
        FROM valid_trips
        WHERE hour BETWEEN 7 AND 20
        GROUP BY date, hour
    ),
    missing_hours AS (
        SELECT d.date, 14 - count(s.hour) AS missing_service_hours
        FROM daily AS d
        LEFT JOIN service_hours AS s ON s.date = d.date
        GROUP BY d.date
    ),
    base AS (
        SELECT
            d.date,
            year(d.date)                       AS year,
            date_trunc('month', d.date)::DATE  AS month,
            isodow(d.date)                     AS day_of_week,
            dayname(d.date)                    AS day_name,
            isodow(d.date) >= 6                AS is_weekend,
            h.name                             AS holiday_name,
            h.name IS NOT NULL                 AS is_holiday,
            t.day_trips,
            prev.day_trips                     AS prev_week_trips,
            next.day_trips                     AS next_week_trips,
            m.missing_service_hours
        FROM dates AS d
        LEFT JOIN holiday_list AS h ON h.date = d.date
        LEFT JOIN daily AS t ON t.date = d.date
        LEFT JOIN daily AS prev ON prev.date = d.date - 7
        LEFT JOIN daily AS next ON next.date = d.date + 7
        LEFT JOIN missing_hours AS m ON m.date = d.date
    )
    SELECT
        date,
        year,
        month,
        day_of_week,
        day_name,
        is_weekend,
        holiday_name,
        is_holiday,
        NOT is_weekend AND NOT is_holiday AS is_workday,
        CASE
            WHEN NOT is_weekend AND NOT is_holiday THEN 'workday'
            ELSE 'weekend_holiday'
        END AS day_type,
        CASE
            WHEN date < DATE '2020-03-09' THEN 'pre'
            WHEN date <= DATE '2020-05-31' THEN 'break'
            ELSE 'post'
        END AS covid_period,
        day_trips,
        missing_service_hours,
        CASE
            WHEN day_trips IS NULL THEN 'no_data'
            WHEN date IN ({strike_sql}) THEN 'strike'
            WHEN missing_service_hours > 0 THEN 'partial'
            WHEN NOT is_holiday
                 AND day_trips < 0.5 * prev_week_trips
                 AND day_trips < 0.5 * next_week_trips THEN 'partial'
            ELSE 'normal'
        END AS day_status
    FROM base
    ORDER BY date
"""

with stage("02.calendar"):
    con.execute(calendar_query)

con.sql("""
    SELECT day_status, count(*) AS days
    FROM calendar
    GROUP BY day_status
    ORDER BY days DESC
""").df()

[perf] 02.calendar: 0.44s | 18.74 cores | peak RAM 0.427 GB | CPU 99.5%


,day_status,days
0,normal,4486
1,no_data,9
2,strike,7
3,partial,3


วันที่ไม่ใช่ normal ทั้งหมด ตรวจด้วยตาว่าสมเหตุสมผลไหม

In [13]:
con.sql("""
    SELECT date, day_name, holiday_name, day_trips, missing_service_hours, day_status
    FROM calendar
    WHERE day_status <> 'normal'
    ORDER BY date
""").df()

,date,day_name,holiday_name,day_trips,missing_service_hours,day_status
0,2011-10-14,Friday,NaN,54738.0,12,partial
1,2011-10-15,Saturday,NaN,NaN,<NA>,no_data
2,2011-10-16,Sunday,NaN,3729.0,4,partial
3,2011-11-05,Saturday,NaN,NaN,<NA>,no_data
4,2012-11-03,Saturday,NaN,NaN,<NA>,no_data
5,2013-07-01,Monday,NaN,3265.0,4,strike
6,2013-07-02,Tuesday,NaN,8.0,12,strike
7,2013-07-03,Wednesday,NaN,6.0,11,strike
8,2013-07-04,Thursday,Independence Day,2.0,13,strike
9,2013-07-05,Friday,NaN,79322.0,3,strike


In [14]:
day_count = fetch_value(con, "SELECT count(*) FROM calendar")
no_data_days = fetch_value(con, "SELECT count(*) FROM calendar WHERE day_status = 'no_data'")
strike_count = fetch_value(con, "SELECT count(*) FROM calendar WHERE day_status = 'strike'")
calendar_trips = fetch_value(con, "SELECT sum(day_trips) FROM calendar")

expected_days = (last_date - first_date).days + 1

print("days in calendar:", day_count, "expected:", expected_days)
print("no_data days    :", no_data_days)
print("strike days     :", strike_count)

# 2013-10-19 and 2013-10-20 are strike days with no data in the feed, they stay no_data
assert day_count == expected_days
assert no_data_days == 9
assert strike_count == len(strike_days) - 2
assert calendar_trips == valid_trips

days in calendar: 4505 expected: 4505
no_data days    : 9
strike days     : 7


In [15]:
holiday_examples = con.sql("""
    SELECT date, holiday_name, is_holiday, is_workday, day_type
    FROM calendar
    WHERE date IN (DATE '2019-11-28', DATE '2019-11-29', DATE '2019-10-14', DATE '2023-01-02')
    ORDER BY date
""").df()

holiday_examples

,date,holiday_name,is_holiday,is_workday,day_type
0,2019-10-14,NaN,False,True,workday
1,2019-11-28,Thanksgiving Day,True,False,weekend_holiday
2,2019-11-29,Day after Thanksgiving,True,False,weekend_holiday
3,2023-01-02,New Year's Day,True,False,weekend_holiday


In [16]:
example_rows = holiday_examples.set_index("date")

assert example_rows.loc[pd.Timestamp("2019-11-28"), "holiday_name"] == "Thanksgiving Day"
assert example_rows.loc[pd.Timestamp("2019-11-29"), "holiday_name"] == "Day after Thanksgiving"
assert not example_rows.loc[pd.Timestamp("2019-10-14"), "is_holiday"]
assert example_rows.loc[pd.Timestamp("2019-10-14"), "day_type"] == "workday"
assert example_rows.loc[pd.Timestamp("2023-01-02"), "holiday_name"] == "New Year's Day"

print("holiday examples ok")

holiday examples ok


## station_hour

ตาราง สถานี × วัน × ชั่วโมง ครบทุกช่อง ตั้งแต่วันเปิดของแต่ละสถานี
- ชั่วโมงที่ไม่มีทริปใส่ 0
- วัน no_data ใส่ค่าว่าง

In [17]:
con.execute("DROP TYPE IF EXISTS station")

station_codes = []
for code in sorted(STATIONS):
    station_codes.append(f"'{code}'")
con.execute(f"CREATE TYPE station AS ENUM ({', '.join(station_codes)})")

station_hour_query = """
    CREATE OR REPLACE TEMP TABLE station_hour AS
    WITH entries AS (
        SELECT origin AS station, date, hour, sum(trips) AS entries
        FROM valid_trips
        GROUP BY origin, date, hour
    ),
    exits AS (
        SELECT dest AS station, date, hour, sum(trips) AS exits
        FROM valid_trips
        GROUP BY dest, date, hour
    ),
    hours AS (
        SELECT range::UTINYINT AS hour
        FROM range(24)
    ),
    grid AS (
        SELECT s.station, c.date, h.hour, c.day_status
        FROM stations AS s
        JOIN calendar AS c ON c.date >= s.opening_date
        CROSS JOIN hours AS h
    )
    SELECT
        g.station::station AS station,
        g.date,
        g.hour,
        CASE
            WHEN g.day_status = 'no_data' THEN NULL
            ELSE coalesce(e.entries, 0)
        END::UINTEGER AS entries,
        CASE
            WHEN g.day_status = 'no_data' THEN NULL
            ELSE coalesce(x.exits, 0)
        END::UINTEGER AS exits
    FROM grid AS g
    LEFT JOIN entries AS e
        ON e.station = g.station AND e.date = g.date AND e.hour = g.hour
    LEFT JOIN exits AS x
        ON x.station = g.station AND x.date = g.date AND x.hour = g.hour
"""

with stage("02.station_hour"):
    con.execute(station_hour_query)

con.sql("SELECT * FROM station_hour ORDER BY date, hour, station LIMIT 5").df()

[perf] 02.station_hour: 1.64s | 17.78 cores | peak RAM 1.572 GB | CPU 92.0%


,station,date,hour,entries,exits
0,12TH,2011-01-01,0,74,127
1,16TH,2011-01-01,0,314,263
2,19TH,2011-01-01,0,111,115
3,24TH,2011-01-01,0,184,293
4,ASHB,2011-01-01,0,51,80


In [18]:
expected_rows = fetch_value(con, """
    SELECT count(*) * 24
    FROM stations AS s
    JOIN calendar AS c ON c.date >= s.opening_date
""")
expected_null_rows = fetch_value(con, """
    SELECT count(*) * 24
    FROM stations AS s
    JOIN calendar AS c ON c.date >= s.opening_date
    WHERE c.day_status = 'no_data'
""")

check = fetch_row(con, """
    SELECT
        count(*),
        count(DISTINCT (station, date, hour)),
        count(*) FILTER (WHERE entries IS NULL),
        count(*) FILTER (WHERE entries = 0),
        sum(entries),
        sum(exits)
    FROM station_hour
""")
rows, unique_keys, null_rows, zero_rows, entries_total, exits_total = check

print(f"rows        : {rows:,} (expected {expected_rows:,})")
print(f"unique keys : {unique_keys:,}")
print(f"null rows   : {null_rows:,} (expected {expected_null_rows:,})")
print(f"zero rows   : {zero_rows:,} ({zero_rows / rows:.1%} filled with 0)")
print(f"entries     : {entries_total:,}")
print(f"exits       : {exits_total:,}")
print(f"valid trips : {valid_trips:,}")

assert rows == expected_rows
assert unique_keys == rows
assert null_rows == expected_null_rows
assert entries_total == valid_trips
assert exits_total == valid_trips

rows        : 5,020,776 (expected 5,020,776)
unique keys : 5,020,776
null rows   : 9,888 (expected 9,888)
zero rows   : 741,446 (14.8% filled with 0)
entries     : 1,194,453,535
exits       : 1,194,453,535
valid trips : 1,194,453,535


สุ่มตัวอย่าง 5 แถวในวันปกติ แล้วคำนวณใหม่จาก fact table ตรงๆ ต้องได้ค่าเดียวกัน

In [19]:
sample_rows = con.sql("""
    SELECT h.station::VARCHAR AS station, h.date, h.hour, h.entries, h.exits
    FROM station_hour AS h
    JOIN calendar AS c ON c.date = h.date
    WHERE c.day_status = 'normal' AND h.entries > 0
    USING SAMPLE 5 ROWS (reservoir, 42)
""").fetchall()

checked = []
for station, date, hour, entries, exits in sample_rows:
    entries_from_fact = fetch_value(con, f"""
        SELECT coalesce(sum(trips), 0) FROM od
        WHERE origin = '{station}' AND date = DATE '{date}' AND hour = {hour}
    """)
    exits_from_fact = fetch_value(con, f"""
        SELECT coalesce(sum(trips), 0) FROM od
        WHERE dest = '{station}' AND date = DATE '{date}' AND hour = {hour}
    """)
    checked.append({
        "station": station,
        "date": date,
        "hour": hour,
        "entries": entries,
        "entries_from_fact": entries_from_fact,
        "exits": exits,
        "exits_from_fact": exits_from_fact,
    })

pd.DataFrame(checked)

,station,date,hour,entries,entries_from_fact,exits,exits_from_fact
0,ASHB,2013-06-15,11,242,242,156,156
1,COLS,2013-05-14,20,174,174,250,250
2,16TH,2020-05-28,17,201,201,85,85
3,WOAK,2012-11-20,8,1084,1084,123,123
4,NBRK,2018-03-14,4,9,9,2,2


In [20]:
for row in checked:
    assert row["entries"] == row["entries_from_fact"]
    assert row["exits"] == row["exits_from_fact"]

print("sample rows match the fact table")

sample rows match the fact table


## station_day และ system_hour

station_day รวมจาก station_hour (วัน no_data ยังเป็นค่าว่าง)
system_hour เป็นทริปทั้งระบบ วัน × ชั่วโมง เติมศูนย์แบบเดียวกัน

In [21]:
station_day_query = """
    CREATE OR REPLACE TEMP TABLE station_day AS
    SELECT
        station,
        date,
        sum(entries)::UINTEGER AS entries,
        sum(exits)::UINTEGER   AS exits
    FROM station_hour
    GROUP BY station, date
"""

system_hour_query = """
    CREATE OR REPLACE TEMP TABLE system_hour AS
    WITH trips_by_hour AS (
        SELECT date, hour, sum(trips) AS trips
        FROM valid_trips
        GROUP BY date, hour
    ),
    hours AS (
        SELECT range::UTINYINT AS hour
        FROM range(24)
    )
    SELECT
        c.date,
        h.hour,
        CASE
            WHEN c.day_status = 'no_data' THEN NULL
            ELSE coalesce(t.trips, 0)
        END::UINTEGER AS trips
    FROM calendar AS c
    CROSS JOIN hours AS h
    LEFT JOIN trips_by_hour AS t ON t.date = c.date AND t.hour = h.hour
"""

with stage("02.station_day_system_hour"):
    con.execute(station_day_query)
    con.execute(system_hour_query)

[perf] 02.station_day_system_hour: 0.64s | 17.89 cores | peak RAM 1.561 GB | CPU 95.1%


In [22]:
station_day_check = fetch_row(con, """
    SELECT count(*), count(DISTINCT (station, date)), sum(entries), sum(exits)
    FROM station_day
""")
system_hour_check = fetch_row(con, """
    SELECT count(*), count(DISTINCT (date, hour)), sum(trips), count(*) FILTER (WHERE trips IS NULL)
    FROM system_hour
""")

sd_rows, sd_keys, sd_entries, sd_exits = station_day_check
sh_rows, sh_keys, sh_trips, sh_nulls = system_hour_check

print(f"station_day : {sd_rows:,} rows, entries {sd_entries:,}, exits {sd_exits:,}")
print(f"system_hour : {sh_rows:,} rows, trips {sh_trips:,}, null rows {sh_nulls}")

assert sd_rows == sd_keys
assert sd_entries == valid_trips
assert sd_exits == valid_trips
assert sh_rows == sh_keys
assert sh_rows == day_count * 24
assert sh_trips == valid_trips
assert sh_nulls == no_data_days * 24

station_day : 209,199 rows, entries 1,194,453,535, exits 1,194,453,535
system_hour : 108,120 rows, trips 1,194,453,535, null rows 216


## od_month และ month_days

od_month ใช้เฉพาะวัน normal แยกวันทำงาน (workday) กับวันหยุด (weekend_holiday)
เติมศูนย์ทุกคู่สถานีที่ทั้งสองสถานีเปิดแล้วภายในเดือนนั้น

ค่าเฉลี่ยต่อวัน = trips / days จาก month_days
ข้อควรระวัง: เดือนที่สถานีเพิ่งเปิดกลางเดือน ค่าเฉลี่ยต่อวันของสถานีนั้นจะต่ำกว่าจริง เพราะหารด้วยจำนวนวันทั้งเดือน

In [23]:
month_days_query = """
    CREATE OR REPLACE TEMP TABLE month_days AS
    SELECT month, day_type, count(*)::USMALLINT AS days
    FROM calendar
    WHERE day_status = 'normal'
    GROUP BY month, day_type
"""

od_month_query = """
    CREATE OR REPLACE TEMP TABLE od_month AS
    WITH trips_by_month AS (
        SELECT
            c.month,
            c.day_type,
            t.origin,
            t.dest,
            sum(t.trips) AS trips
        FROM valid_trips AS t
        JOIN calendar AS c ON c.date = t.date
        WHERE c.day_status = 'normal'
        GROUP BY c.month, c.day_type, t.origin, t.dest
    ),
    grid AS (
        SELECT m.month, m.day_type, o.station AS origin, d.station AS dest
        FROM month_days AS m
        JOIN stations AS o ON o.opening_date <= last_day(m.month)
        JOIN stations AS d ON d.opening_date <= last_day(m.month)
    )
    SELECT
        g.month,
        g.day_type,
        g.origin::station AS origin,
        g.dest::station   AS dest,
        coalesce(t.trips, 0)::UINTEGER AS trips
    FROM grid AS g
    LEFT JOIN trips_by_month AS t
        ON t.month = g.month
       AND t.day_type = g.day_type
       AND t.origin = g.origin
       AND t.dest = g.dest
"""

with stage("02.od_month"):
    con.execute(month_days_query)
    con.execute(od_month_query)

[perf] 02.od_month: 0.98s | 17.7 cores | peak RAM 1.514 GB | CPU 99.5%


In [24]:
normal_day_trips = fetch_value(con, """
    SELECT sum(t.trips)
    FROM valid_trips AS t
    JOIN calendar AS c ON c.date = t.date
    WHERE c.day_status = 'normal'
""")
normal_days = fetch_value(con, "SELECT count(*) FROM calendar WHERE day_status = 'normal'")

od_check = fetch_row(con, """
    SELECT
        count(*),
        count(DISTINCT (month, day_type, origin, dest)),
        sum(trips),
        count(*) FILTER (WHERE trips = 0)
    FROM od_month
""")
od_rows, od_keys, od_trips, od_zero_rows = od_check
month_day_total = fetch_value(con, "SELECT sum(days) FROM month_days")

print(f"od_month   : {od_rows:,} rows, {od_trips:,} trips, {od_zero_rows:,} rows filled with 0")
print(f"month_days : {month_day_total:,} days (normal days in calendar: {normal_days:,})")

assert od_rows == od_keys
assert od_trips == normal_day_trips
assert month_day_total == normal_days

od_month   : 643,286 rows, 1,194,200,880 trips, 1,089 rows filled with 0
month_days : 4,486 days (normal days in calendar: 4,486)


## สรุปการตรวจ

ทุก mart ต้องมีผลรวมทริปตรงกับ fact table หลังตัดทริปหลงก่อนวันเปิดสถานี
ส่วน od_month ใช้เฉพาะวัน normal จึงน้อยกว่าทริปในวัน strike และ partial

In [25]:
non_normal_trips = valid_trips - normal_day_trips

totals = pd.DataFrame([
    {"table": "fact table (od)",          "trips": fact_trips},
    {"table": "stray trips removed",      "trips": -stray_trips},
    {"table": "valid trips",              "trips": valid_trips},
    {"table": "station_hour entries",     "trips": entries_total},
    {"table": "station_hour exits",       "trips": exits_total},
    {"table": "station_day entries",      "trips": sd_entries},
    {"table": "system_hour",              "trips": sh_trips},
    {"table": "calendar day_trips",       "trips": calendar_trips},
    {"table": "trips on strike/partial",  "trips": -non_normal_trips},
    {"table": "od_month (normal days)",   "trips": od_trips},
])

totals

,table,trips
0,fact table (od),1194454788
1,stray trips removed,-1253
2,valid trips,1194453535
3,station_hour entries,1194453535
4,station_hour exits,1194453535
5,station_day entries,1194453535
6,system_hour,1194453535
7,calendar day_trips,1194453535
8,trips on strike/partial,-252655
9,od_month (normal days),1194200880


## เขียน marts เป็น Parquet

เรียงตาม key ของแต่ละตาราง บีบอัด zstd

In [26]:
def write_mart(con, name, order_by):
    path = MARTS / f"{name}.parquet"
    con.execute(f"""
        COPY (SELECT * FROM {name} ORDER BY {order_by})
        TO '{path}' (FORMAT parquet, COMPRESSION zstd, COMPRESSION_LEVEL 6)
    """)
    return path


marts = {
    "stations": "station",
    "calendar": "date",
    "station_hour": "date, hour, station",
    "station_day": "date, station",
    "system_hour": "date, hour",
    "od_month": "month, day_type, origin, dest",
    "month_days": "month, day_type",
}

with stage("02.write_marts"):
    con.execute("SET preserve_insertion_order = true")
    for name, order_by in marts.items():
        write_mart(con, name, order_by)
    con.execute("SET preserve_insertion_order = false")

[perf] 02.write_marts: 0.23s | 10.22 cores | peak RAM 1.055 GB | CPU 58.3%


In [27]:
mart_rows = []
for name in marts:
    path = MARTS / f"{name}.parquet"
    rows_in_file = fetch_value(con, f"SELECT count(*) FROM read_parquet('{path}')")
    mart_rows.append({
        "mart": name,
        "rows": rows_in_file,
        "size_mb": round(path.stat().st_size / 1e6, 2),
    })

pd.DataFrame(mart_rows)

,mart,rows,size_mb
0,stations,50,0.00
1,calendar,4505,0.03
2,station_hour,5020776,12.60
3,station_day,209199,1.06
4,system_hour,108120,0.26
5,od_month,643286,1.36
6,month_days,297,0.00


## ดูผลการเติมข้อมูล

ทริปทั้งระบบรายวัน จาก system_hour ช่วง มิ.ย.-พ.ย. 2013 ซึ่งมีทั้ง strike และวันที่ไม่มีข้อมูล
วัน no_data เส้นจะขาด (ไม่ได้ตกลงไปที่ 0) และแรเงาสีเทาไว้ ส่วนวัน strike ทำจุดสีไว้ให้เห็น

In [28]:
daily_status = con.sql("""
    SELECT c.date, c.day_status, sum(s.trips) AS trips
    FROM system_hour AS s
    JOIN calendar AS c ON c.date = s.date
    WHERE c.date BETWEEN DATE '2013-06-01' AND DATE '2013-11-30'
    GROUP BY c.date, c.day_status
    ORDER BY c.date
""").df()

strike_points = daily_status[daily_status["day_status"] == "strike"]
partial_points = daily_status[daily_status["day_status"] == "partial"]

line = go.Scatter(
    x=daily_status["date"],
    y=daily_status["trips"],
    mode="lines",
    name="Trips per day",
    line=dict(color=viz.ACCENT, width=2),
    connectgaps=False,
)
strikes = go.Scatter(
    x=strike_points["date"],
    y=strike_points["trips"],
    mode="markers",
    name="Strike",
    marker=dict(color=viz.CONTRAST, size=8),
)
partials = go.Scatter(
    x=partial_points["date"],
    y=partial_points["trips"],
    mode="markers",
    name="Partial data",
    marker=dict(color=viz.SERIES[3], size=8),
)

fig = go.Figure([line, strikes, partials])
fig.update_layout(hovermode="closest")

no_data_points = daily_status[daily_status["day_status"] == "no_data"]
for day in no_data_points["date"]:
    fig.add_vrect(
        x0=day - pd.Timedelta(hours=12),
        x1=day + pd.Timedelta(hours=12),
        fillcolor=viz.CONTEXT,
        opacity=0.5,
        line_width=0,
    )
fig.update_yaxes(tickformat="~s", title_text="trips per day")

title = "Two BART strikes in 2013, and two days the feed is missing"
subtitle = "Trips per day, Jun to Nov 2013. Grey band = 2013-10-19 and 10-20, no data in the feed (not zero)"
viz.titled(fig, title, subtitle)
viz.save(fig, "02_daily_trips_status")
fig

## สรุป

- สร้าง marts 7 ตาราง ขนาดรวมประมาณ 15 MB ทั้งหมดใช้เวลารวมไม่ถึง 5 วินาที
- วันเปิดสถานีที่หาจากข้อมูลตรงกับวันเปิดทางการครบ 7 สถานี ตัดทริปหลงก่อนวันเปิดออก 462 แถว (1,253 ทริป)
- วันหยุดเลือกจากข้อมูลจริง ไม่นับ Columbus Day, Veterans Day และ Juneteenth เพราะคนยังทำงาน (ผู้โดยสาร 77-100% ของวันปกติ) และเพิ่ม Day after Thanksgiving (48%)
- calendar 4,505 วัน: normal 4,486 วัน, no_data 9 วัน, strike 7 วัน, partial 3 วัน
- strike 2013 ทั้งสองครั้งเห็นชัดในข้อมูล ทั้งวันมีทริปเกือบเป็น 0 และวันที่ขาด 2013-10-19/20 ก็อยู่ในช่วง strike
- station_hour 5,020,776 แถว ครบทุกช่องตั้งแต่วันเปิดของแต่ละสถานี เติม 0 ไป 14.8% วัน no_data เป็นค่าว่าง 9,888 แถว
- ผลรวมทริปของ station_hour, station_day, system_hour และ calendar ตรงกันทุกตารางที่ 1,194,453,535 ทริป
- od_month ใช้เฉพาะวัน normal น้อยกว่ายอดรวม 252,655 ทริป เท่ากับทริปในวัน strike และ partial พอดี

ต่อไป: 03_eda